<a href="https://colab.research.google.com/github/Silva-DTS/Statistical-Learning-e23379/blob/main/Assignment_7c_Item_Response_Prediction_and_Click_Through_Rate_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bayesian Estimation of a User Ability Parameter from Item Responses

The online learning platform presents a user with a sequence of multiple-choice questions and updates its estimate of the user's latent ability after each response.

Let the response to the $i$-th item be

$$
Y_i=
\begin{cases}
1, & \text{if the user answers correctly},\\
0, & \text{if the user answers incorrectly}.
\end{cases}
$$

The probability of answering item $i$ correctly follows the Two-Parameter Logistic (2PL) Item Response Theory model,

$$
P(Y_i=1\mid\Theta=\theta)
=
p_i(\theta)
=
\frac{1}{1+\exp\left[-a_i(\theta-b_i)\right]},
$$

where

- $a_i>0$ is the discrimination parameter,
- $b_i$ is the difficulty parameter,
- $\theta$ is the user's latent ability.

Initially, the user's ability is assumed to follow a standard normal distribution,

$$
\Theta\sim N(0,1),
$$

with prior density

$$
f_\Theta^{(0)}(\theta)
=
\frac{1}{\sqrt{2\pi}}
\exp\left(-\frac{\theta^2}{2}\right).
$$


# Task 1 – Visualizing the Mechanics

The probability that a user answers an item correctly is

$$
p_i(\theta)
=
\frac{1}{1+\exp[-a_i(\theta-b_i)]}.
$$

The discrimination parameter $a_i$ determines the steepness of the logistic curve, while the difficulty parameter $b_i$ determines the horizontal location of the curve.

In [1]:
import numpy as np
import plotly.graph_objects as go

# --------------------------------------------------
# Two-Parameter Logistic (2PL) Item Response Function
# --------------------------------------------------

def p_i(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

# Ability grid
theta_vals = np.linspace(-6, 6, 400)

# Curve configurations
curves = [
    {"a": 0.5, "b": 0},
    {"a": 1.5, "b": -2},
    {"a": 1.5, "b": 0},
    {"a": 1.5, "b": 2},
]

fig = go.Figure()

for curve in curves:

    probabilities = p_i(theta_vals, curve["a"], curve["b"])

    fig.add_trace(
        go.Scatter(
            x=theta_vals,
            y=probabilities,
            mode="lines",
            name=f"a = {curve['a']}, b = {curve['b']}"
        )
    )

fig.update_layout(
    title="Two-Parameter Logistic (2PL) Item Response Curves",
    xaxis_title="Latent Ability (θ)",
    yaxis_title="P(Y = 1 | θ)",
    template="plotly_white"
)

fig.show()

## Interpretation

The graph illustrates the effects of the discrimination parameter $a_i$ and the difficulty parameter $b_i$.

### Effect of the discrimination parameter

The discrimination parameter controls the steepness of the logistic curve.

- A **small value of $a_i$** produces a gradual slope, meaning that the probability changes slowly with ability.
- A **large value of $a_i$** produces a much steeper slope, meaning that small differences in ability produce large differences in the probability of answering correctly.

Thus, highly discriminating items distinguish high-ability users from low-ability users more effectively.

### Effect of the difficulty parameter

The difficulty parameter shifts the curve horizontally.

- Increasing $b_i$ shifts the curve to the **right**.
- Decreasing $b_i$ shifts the curve to the **left**.

Every curve crosses

$$
P(Y_i=1|\Theta=\theta)=0.5
$$

exactly when

$$
\theta=b_i.
$$

Therefore, increasing the difficulty parameter requires a higher ability level before the user has a 50% chance of answering correctly.

# Task 2 – Sequential Likelihood Contribution

Let the current item be

$$
y_k\in\{0,1\}.
$$

The likelihood contribution of this single response is

$$
L(y_k|\theta)
=
[p_k(\theta)]^{y_k}
[1-p_k(\theta)]^{1-y_k}.
$$

Since item responses are conditionally independent given the latent ability $\Theta$, the joint likelihood for the running response history

$$
y^{(k)}
=
(y_1,y_2,\ldots,y_k)
$$

is

$$
L(y^{(k)}|\theta)
=
\prod_{i=1}^{k}
[p_i(\theta)]^{y_i}
[1-p_i(\theta)]^{1-y_i}.
$$

Each newly observed response contributes one additional multiplicative likelihood factor to the running likelihood function.

# Task 3 – Mathematical Formulation of the Running Update

Using Bayes' theorem, the posterior distribution after observing item $k$ is proportional to the product of the previous posterior distribution and the likelihood contribution of the new response.

Ignoring the normalizing constant,

$$
f_{\Theta|Y^{(k)}}(\theta|y^{(k)})
\propto
L(y_k|\theta)
\,
f_{\Theta|Y^{(k-1)}}(\theta|y^{(k-1)}).
$$

Substituting the likelihood function,

$$
f_{\Theta|Y^{(k)}}(\theta|y^{(k)})
\propto
[p_k(\theta)]^{y_k}
[1-p_k(\theta)]^{1-y_k}
f_{\Theta|Y^{(k-1)}}(\theta|y^{(k-1)}).
$$

The fully normalized posterior distribution is

$$
f_{\Theta|Y^{(k)}}(\theta|y^{(k)})
=
\frac{
[p_k(\theta)]^{y_k}
[1-p_k(\theta)]^{1-y_k}
f_{\Theta|Y^{(k-1)}}(\theta|y^{(k-1)})
}
{
\int_{-\infty}^{\infty}
[p_k(s)]^{y_k}
[1-p_k(s)]^{1-y_k}
f_{\Theta|Y^{(k-1)}}(s|y^{(k-1)})\,ds
}.
$$

The denominator is the normalizing constant that ensures the posterior density integrates to one.

For the first item,

$$
f_{\Theta|Y^{(0)}}(\theta)
=
\frac{1}{\sqrt{2\pi}}
\exp\left(-\frac{\theta^2}{2}\right),
$$

which is simply the initial standard normal prior.

Therefore, after every new response, the posterior distribution from the previous step becomes the prior distribution for the next Bayesian update.

# Task 4 – Dynamic Shifting of the Posterior Distribution

Suppose the user answers the current item correctly, that is,

$$
y_k=1,
$$

and the item has a large difficulty parameter $b_k$.

The likelihood contribution of this response becomes

$$
L(y_k=1|\theta)=p_k(\theta).
$$

Since the item is highly difficult, users with low ability have only a small probability of answering it correctly, while users with high ability have a much larger probability.

Mathematically,

$$
p_k(\theta)
=
\frac{1}{1+\exp[-a_k(\theta-b_k)]}
$$

takes relatively large values only for larger values of $\theta$.

The updated posterior is

$$
f_{\Theta|Y^{(k)}}(\theta|y^{(k)})
\propto
p_k(\theta)
f_{\Theta|Y^{(k-1)}}(\theta|y^{(k-1)}).
$$

Multiplying the previous posterior by this likelihood increases the probability density for larger ability values while decreasing it for smaller ability values.

So, the peak (mode) of the posterior distribution shifts toward the right, indicating that the platform now considers the user more likely to possess a higher latent ability than before observing the response.

The magnitude of this shift depends on both the difficulty parameter $b_k$ and the discrimination parameter $a_k$. A correct response to a very difficult and highly discriminating item typically produces a larger rightward shift than a correct response to an easier item.

# Task 5 – Tracking Certainty and Sharpness

The discrimination parameter $a_k$ determines how informative the current item is about the user's ability.

When $a_k$ is very large, the Item Response Function becomes much steeper around the item's difficulty level.

As a result,

- the likelihood function becomes highly concentrated,
- the posterior distribution becomes narrower,
- the posterior variance decreases,
- the estimate of the user's ability becomes more precise.

Therefore, a highly discriminating item produces a sharper posterior distribution and increases the platform's confidence in its estimate.

Conversely, when $a_k$ is very small, the Item Response Function becomes flatter.

In this case,

- users with different ability levels have similar probabilities of answering correctly,
- the likelihood function contains less information,
- the posterior changes only slightly,
- the posterior variance remains relatively large.

Hence, weakly discriminating items contribute less information about the user's ability, resulting in a broader posterior distribution and lower certainty.

In summary,

- **Large $a_k$** → sharper posterior, lower variance, greater confidence.
- **Small $a_k$** → broader posterior, higher variance, lower confidence.

# Task 6 – Numerical Implementation of a Running Grid

For the 2PL model, the posterior distribution generally has no closed-form analytical solution. Therefore, it is approximated numerically on a fixed grid of ability values.

The sequential Bayesian updating algorithm is as follows.

### Step 1

Construct a fixed grid of ability values,

$$
\theta_1,\theta_2,\ldots,\theta_m,
$$

covering a sufficiently wide range (for example, from $-5$ to $5$).

### Step 2

Evaluate the initial prior distribution on this grid,

$$
f_{\Theta}^{(0)}(\theta)
=
\frac{1}{\sqrt{2\pi}}
\exp\left(-\frac{\theta^2}{2}\right).
$$

### Step 3

After observing response $y_k$, compute the probability of a correct response over the entire grid,

$$
p_k(\theta)
=
\frac{1}{1+\exp[-a_k(\theta-b_k)]}.
$$

### Step 4

Compute the likelihood values on the grid,

$$
L(y_k|\theta)
=
[p_k(\theta)]^{y_k}
[1-p_k(\theta)]^{1-y_k}.
$$

### Step 5

Update the posterior distribution by multiplying the previous posterior with the likelihood,

$$
f_{\text{new}}(\theta)
=
f_{\text{old}}(\theta)
\times
L(y_k|\theta).
$$

### Step 6

Normalize the posterior numerically so that it integrates to one,

$$
f_{\text{new}}(\theta)
=
\frac{
f_{\text{new}}(\theta)
}{
\int
f_{\text{new}}(\theta)
\,d\theta
}.
$$

Using the trapezoidal rule, the normalization step is implemented as

```python
posterior *= likelihood

posterior /= np.trapezoid(posterior, theta_grid)
```

The function `np.trapezoid()` numerically approximates the integral of the posterior density over the ability grid, ensuring that the total probability equals one.

### Step 7

After normalization, compute the Posterior Mean (Bayesian estimate),

$$
\hat{\theta}_{\text{Bayes}}
=
\int
\theta
f(\theta|y)
\,d\theta,
$$

which is approximated numerically by

```python
theta_bayes = np.trapezoid(theta_grid * posterior, theta_grid)
```

The Maximum A Posteriori (MAP) estimate is obtained by locating the grid point with the largest posterior density,

$$
\hat{\theta}_{MAP}
=
\underset{\theta}{\operatorname{argmax}}
\;
f(\theta|y),
$$

implemented as

```python
theta_map = theta_grid[np.argmax(posterior)]
```

These estimates are stored after every response, and the posterior distribution becomes the prior distribution for the next Bayesian update.

# Task 7 – Evaluating Convergence over the Timeline

user's true latent ability is

$$
\theta_{\text{true}} = 0.75.
$$

The objective is to examine how the Bayesian estimators evolve as the user answers a sequence of questions.

The simulation consists of the following steps:

- Generate a sequence of $n=20$ items.
- For each item, randomly generate its discrimination and difficulty parameters according to

$$
a_k \sim \text{Uniform}(0.5,2.0),
$$

and

$$
b_k \sim N(0,1).
$$

- Compute the probability that a user with true ability $\theta_{\text{true}}$ answers the item correctly,

$$
p_k(\theta_{\text{true}})
=
\frac{1}
{1+\exp[-a_k(\theta_{\text{true}}-b_k)]}.
$$

- Simulate the user's response by comparing this probability with a random draw from the Uniform distribution,

$$
U\sim \text{Uniform}(0,1).
$$

The simulated response is

$$
Y_k=
\begin{cases}
1, & U<p_k(\theta_{\text{true}}),\\
0, & \text{otherwise}.
\end{cases}
$$

After each response, the posterior distribution is updated sequentially. The following two estimates are computed after every item:

### Posterior Mean (Bayes Estimate)

$$
\hat{\theta}_{\text{Bayes}}
=
\int_{-\infty}^{\infty}
\theta
f(\theta|y)
\,d\theta.
$$

### Maximum A Posteriori (MAP) Estimate

$$
\hat{\theta}_{MAP}
=
\underset{\theta}{\operatorname{argmax}}
\;
f(\theta|y).
$$

In [2]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# ----------------------------------------------------------
# Sequential Bayesian Ability Estimation using Grid Approximation
# ----------------------------------------------------------

np.random.seed(42)

# True latent ability
theta_true = 0.75

# Number of items
n_items = 20

# Ability grid
theta_grid = np.linspace(-5, 5, 1000)

# Initial prior distribution
posterior = stats.norm.pdf(theta_grid, 0, 1)

# 2PL Item Response Function
def p_i(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

# Randomly generate item parameters
a_params = np.random.uniform(0.5, 2.0, n_items)
b_params = np.random.normal(0, 1, n_items)

# Store estimates
posterior_mean = [0.0]
map_estimate = [0.0]

steps = np.arange(n_items + 1)

# ----------------------------------------------------------
# Sequential Bayesian Updating
# ----------------------------------------------------------

for k in range(n_items):

    a = a_params[k]
    b = b_params[k]

    # Probability of correct response for the true ability
    p_true = p_i(theta_true, a, b)

    # Simulated response
    y = 1 if np.random.rand() < p_true else 0

    # Likelihood evaluated over the grid
    p_grid = p_i(theta_grid, a, b)

    likelihood = (p_grid ** y) * ((1 - p_grid) ** (1 - y))

    # Bayesian update
    posterior *= likelihood

    # Normalize
    posterior /= np.trapezoid(posterior, theta_grid)

    # Posterior Mean
    theta_bayes = np.trapezoid(theta_grid * posterior, theta_grid)

    # MAP Estimate
    theta_map = theta_grid[np.argmax(posterior)]

    posterior_mean.append(theta_bayes)
    map_estimate.append(theta_map)

# ----------------------------------------------------------
# Plot Convergence
# ----------------------------------------------------------

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=steps,
        y=posterior_mean,
        mode="lines+markers",
        name="Posterior Mean"
    )
)

fig.add_trace(
    go.Scatter(
        x=steps,
        y=map_estimate,
        mode="lines+markers",
        name="MAP Estimate"
    )
)

fig.add_hline(
    y=theta_true,
    line_dash="dash",
    line_color="red",
    annotation_text="True Ability"
)

fig.update_layout(
    title="Convergence of Bayesian Ability Estimates",
    xaxis_title="Item Number",
    yaxis_title="Estimated Ability",
    template="plotly_white"
)

fig.show()

## Analysis

Initially, both the Posterior Mean and the MAP estimate are equal to the mean of the prior distribution, which is zero. At this stage, the platform has no information about the user's true ability other than the prior assumption.

As more responses are observed, each new item contributes additional information through its likelihood function. Consequently, both estimators gradually move toward the true latent ability,

$$
\theta_{\text{true}} = 0.75.
$$

Small fluctuations may occur because the responses are generated randomly. For example, an incorrect response to an easy item or a correct response to a difficult item may temporarily shift the estimates away from the true ability. However, as the number of observed items increases, these random fluctuations become less influential.

The posterior distribution also becomes progressively narrower after successive updates. This reduction in posterior variance indicates that the platform becomes increasingly confident about its estimate of the user's ability.

Therefore, the distance between both estimators and the true latent ability generally decreases as the number of answered items increases. This demonstrates that sequential Bayesian updating enables the learning platform to estimate the user's latent ability with increasing accuracy and confidence over time.

# Q. Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates

The e-commerce platform wishes to estimate the click-through rate (CTR) of a newly launched advertisement as user interactions are observed sequentially. Instead of waiting until a large number of impressions have accumulated, the platform updates its estimate immediately after each user interaction.

Let $\Theta$ denote the unknown click-through rate (CTR), where

$$
0 \leq \Theta \leq 1.
$$

For each advertisement impression, the observed user interaction is represented by

$$
Y_k=
\begin{cases}
1, & \text{if the user clicks the advertisement},\\
0, & \text{if the user does not click the advertisement}.
\end{cases}
$$

Conditioned on the true click probability $\Theta=\theta$, every user interaction is assumed to be an independent Bernoulli trial with

$$
P(Y_k=1\mid\Theta=\theta)=\theta.
$$

Before observing any user interactions, the unknown click probability is assigned a Beta prior distribution,

$$
\Theta\sim\mathrm{Beta}(\alpha_0,\beta_0),
$$

whose probability density function is

$$
f_{\Theta}^{(0)}(\theta)
=
\frac{1}{B(\alpha_0,\beta_0)}
\theta^{\alpha_0-1}
(1-\theta)^{\beta_0-1},
\qquad
0\le\theta\le1,
$$

where $B(\alpha,\beta)$ is the Beta function that acts as the normalizing constant.

Since the Beta distribution is conjugate to the Bernoulli likelihood, the posterior distribution after each observed user interaction remains a Beta distribution. Therefore, the posterior obtained after one impression becomes the prior distribution for the next impression, enabling efficient sequential Bayesian updating.

# Task 1 – Structural Probability and Properties

The probability density function of a Beta distribution is

$$
f(\theta)
=
\frac{1}{B(\alpha,\beta)}
\theta^{\alpha-1}
(1-\theta)^{\beta-1},
\qquad
0\le\theta\le1.
$$

 consider the following three Beta distributions to get the shape:

- **Uninformative prior:** $\mathrm{Beta}(1,1)$
- **Right-skewed distribution:** $\mathrm{Beta}(2,8)$
- **Left-skewed distribution:** $\mathrm{Beta}(8,2)$

The following Plotly visualization compares these probability density functions.

In [3]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# Generate theta values over [0,1]
theta = np.linspace(0,1,500)

# Beta distributions
configs = [
    {"alpha":1,"beta":1,"name":"Beta(1,1)"},
    {"alpha":2,"beta":8,"name":"Beta(2,8)"},
    {"alpha":8,"beta":2,"name":"Beta(8,2)"}
]

fig = go.Figure()

for config in configs:

    pdf = stats.beta.pdf(theta,
                         config["alpha"],
                         config["beta"])

    fig.add_trace(
        go.Scatter(
            x=theta,
            y=pdf,
            mode="lines",
            name=config["name"]
        )
    )

fig.update_layout(
    title="Probability Density Functions of Beta Distributions",
    xaxis_title="Click-Through Rate (θ)",
    yaxis_title="Probability Density",
    template="plotly_white"
)

fig.show()

## Interpretation

The Beta distribution is controlled by two positive shape parameters, $\alpha$ and $\beta$, which determine where the probability mass is concentrated over the interval $[0,1]$.

### Beta(1,1)

When

$$
\alpha=\beta=1,
$$

the distribution is uniform over the interval $[0,1]$. Every value of $\theta$ is considered equally likely, representing complete initial uncertainty.

### Beta(2,8)

Since

$$
\beta>\alpha,
$$

most of the probability mass is concentrated near zero. This indicates a prior belief that the advertisement has a relatively low click-through rate.

### Beta(8,2)

Since

$$
\alpha>\beta,
$$

the density shifts toward one, indicating a prior belief that the advertisement is likely to achieve a high click-through rate.

Therefore,

- increasing $\alpha$ shifts the center of mass toward larger values of $\theta$,
- increasing $\beta$ shifts the center of mass toward smaller values of $\theta$.

The balance between these two parameters determines both the location and the shape of the Beta distribution.

# Task 2 – Sequential Likelihood and Joint History

At impression step $k$, the observed user interaction

$$
y_k\in\{0,1\}
$$

follows a Bernoulli distribution.

The likelihood contribution of a single observation is therefore

$$
L(y_k\mid\theta)
=
\theta^{y_k}
(1-\theta)^{1-y_k}.
$$

If

$$
y_k=1,
$$

then

$$
L(y_k\mid\theta)=\theta.
$$

If

$$
y_k=0,
$$

then

$$
L(y_k\mid\theta)=1-\theta.
$$

Assuming that all user interactions are conditionally independent given $\Theta$, the joint likelihood for the running history

$$
y^{(k)}
=
(y_1,y_2,\ldots,y_k)
$$

is

$$
L(y^{(k)}\mid\theta)
=
\prod_{i=1}^{k}
\theta^{y_i}
(1-\theta)^{1-y_i}.
$$

Let

$$
C_k=\sum_{i=1}^{k}y_i
$$

denote the total number of clicks observed after $k$ impressions.

The number of non-clicks is therefore

$$
k-C_k.
$$

Hence, the joint likelihood can be written more compactly as

$$
L(y^{(k)}\mid\theta)
=
\theta^{C_k}
(1-\theta)^{k-C_k}.
$$



# Task 3 – Closed-Form Analytical Updates (Conjugacy)

Using Bayes' theorem, the posterior distribution after observing the $k$-th interaction is

$$
f_{\Theta|Y^{(k)}}(\theta|y^{(k)})
=
\frac{
L(y_k|\theta)
f_{\Theta|Y^{(k-1)}}(\theta|y^{(k-1)})
}{
\int_0^1
L(y_k|s)
f_{\Theta|Y^{(k-1)}}(s|y^{(k-1)})
\,ds
}.
$$

Ignoring the normalizing constant,

$$
f_{\Theta|Y^{(k)}}(\theta|y^{(k)})
\propto
L(y_k|\theta)
f_{\Theta|Y^{(k-1)}}(\theta|y^{(k-1)}).
$$

Assume that the prior at step $k$ is

$$
\Theta
\sim
\mathrm{Beta}
(\alpha_{k-1},\beta_{k-1}),
$$

whose density is

$$
f_{\Theta|Y^{(k-1)}}(\theta)
\propto
\theta^{\alpha_{k-1}-1}
(1-\theta)^{\beta_{k-1}-1}.
$$

Substituting the Bernoulli likelihood,

$$
L(y_k|\theta)
=
\theta^{y_k}
(1-\theta)^{1-y_k},
$$

gives

$$
f_{\Theta|Y^{(k)}}(\theta)
\propto
\theta^{y_k}
(1-\theta)^{1-y_k}
\theta^{\alpha_{k-1}-1}
(1-\theta)^{\beta_{k-1}-1}.
$$

Combining the exponents,

$$
f_{\Theta|Y^{(k)}}(\theta)
\propto
\theta^{\alpha_{k-1}+y_k-1}
(1-\theta)^{\beta_{k-1}+1-y_k-1}.
$$

This has exactly the same functional form as a Beta distribution. Therefore,

$$
\Theta|Y^{(k)}
\sim
\mathrm{Beta}
(\alpha_k,\beta_k),
$$

where the updated parameters are

$$
\boxed{
\alpha_k=\alpha_{k-1}+y_k
}
$$

and

$$
\boxed{
\beta_k=\beta_{k-1}+(1-y_k).
}
$$

Hence, the Beta distribution is conjugate to the Bernoulli likelihood.

The posterior mean at step $k$ is therefore

$$
E[\Theta|Y^{(k)}]
=
\frac{\alpha_k}
{\alpha_k+\beta_k}.
$$

Using the cumulative number of clicks,

$$
\alpha_k
=
\alpha_0+C_k,
$$

and

$$
\beta_k
=
\beta_0+k-C_k,
$$

the posterior mean may also be written as

$$
E[\Theta|Y^{(k)}]
=
\frac{\alpha_0+C_k}
{\alpha_0+\beta_0+k}.
$$

This expression shows that the posterior mean combines the information from the prior distribution with the observed data. As the number of impressions increases, the influence of the prior gradually decreases, and the posterior mean approaches the observed click-through rate.

# Task 4 – Dynamic Shifting Mechanics

Each newly observed user interaction updates the Beta posterior by modifying its shape parameters. Since the posterior distribution remains within the Beta family, these updates are obtained analytically without requiring numerical approximation.

## Effect of an Observed Click

Suppose the observed response is

$$
y_k=1.
$$

The update equations become

$$
\alpha_k=\alpha_{k-1}+1,
$$

$$
\beta_k=\beta_{k-1}.
$$

Increasing $\alpha$ while keeping $\beta$ unchanged shifts the probability density toward larger values of $\theta$. Consequently, the posterior distribution assigns greater probability to higher click-through rates, reflecting increased evidence that the advertisement performs well.

When both

$$
\alpha_k>1
\quad\text{and}\quad
\beta_k>1,
$$

the mode (peak) of the Beta distribution is

$$
\theta_{\mathrm{MAP}}
=
\frac{\alpha_k-1}
{\alpha_k+\beta_k-2},
$$

which moves toward the right after a click.

---

## Effect of a Non-Click

Suppose the observed response is

$$
y_k=0.
$$

The update equations become

$$
\alpha_k=\alpha_{k-1},
$$

$$
\beta_k=\beta_{k-1}+1.
$$

Increasing $\beta$ shifts the posterior density toward smaller values of $\theta$, indicating increased evidence that the advertisement has a lower click-through rate.

Consequently, the posterior mode shifts toward the left.

---

## Comparison with Non-Conjugate Models

The Beta-Binomial model is an example of a **conjugate Bayesian model**.

Because the Beta prior and Bernoulli likelihood are conjugate,

- the posterior remains a Beta distribution,
- the parameters are updated using simple arithmetic,
- no numerical integration is required.

The recursive updates are simply

$$
\alpha_k=\alpha_{k-1}+y_k,
$$

$$
\beta_k=\beta_{k-1}+(1-y_k).
$$

In contrast, models such as the Two-Parameter Logistic (2PL) Item Response Theory model do not possess a conjugate prior.

In the 2PL model,

- the posterior distribution has no closed-form expression,
- Bayesian updating requires numerical approximation,
- posterior normalization is performed using numerical integration over a grid of parameter values.

Therefore, the Beta-Binomial model is computationally much more efficient for sequential updating because every posterior distribution is obtained analytically rather than numerically.

# Task 5 – Running Point Estimators

Since the posterior distribution after each update is

$$
\Theta|Y^{(k)}
\sim
\mathrm{Beta}(\alpha_k,\beta_k),
$$

both the Bayesian estimate and the Maximum A Posteriori (MAP) estimate can be evaluated directly from the updated shape parameters.

## Running Posterior Mean (Bayes Estimate)

The posterior mean is

$$
\boxed{
\hat{\theta}^{(k)}_{\mathrm{Bayes}}
=
E[\Theta|Y^{(k)}]
=
\frac{\alpha_k}
{\alpha_k+\beta_k}
}
$$

This estimator minimizes the expected squared-error loss and represents the average of the posterior distribution.

---

## Running Maximum A Posteriori (MAP) Estimate

For a Beta distribution with

$$
\alpha_k>1
\quad\text{and}\quad
\beta_k>1,
$$

the MAP estimate is

$$
\boxed{
\hat{\theta}^{(k)}_{\mathrm{MAP}}
=
\frac{\alpha_k-1}
{\alpha_k+\beta_k-2}
}
$$

This estimate corresponds to the value of $\theta$ where the posterior density reaches its maximum.

---

## Boundary Cases

When either

$$
\alpha_k\le1
$$

or

$$
\beta_k\le1,
$$

the Beta distribution does not possess an interior mode.

In these situations,

- if $\alpha_k\le1<\beta_k$, the mode occurs at

$$
\theta=0,
$$

- if $\beta_k\le1<\alpha_k$, the mode occurs at

$$
\theta=1,
$$

- if

$$
\alpha_k=\beta_k=1,
$$

the posterior distribution is uniform over the interval $[0,1]$, and every value of $\theta$ is equally likely.

These boundary conditions are important during the first few sequential updates when only a small number of user interactions have been observed.

---

## Interpretation

The Posterior Mean incorporates information from both the prior distribution and the observed data, producing a smooth estimate of the click-through rate.

The MAP estimate identifies the most probable value of the click-through rate according to the posterior distribution.

As the number of impressions increases, both estimators gradually converge toward the true click-through rate because the accumulated evidence increasingly dominates the influence of the initial prior.

# Task 6 – Performance Tracking and Convergence Analysis

The objective is to simulate sequential Bayesian estimation of an advertisement's click-through rate (CTR) over $100$ user impressions.

The true hidden CTR is assumed to be

$$
\theta_{\text{true}}=0.35.
$$

The initial uncertainty is represented using a uniform Beta prior,

$$
\Theta\sim\mathrm{Beta}(1,1),
$$

which means that every possible value of $\theta$ in the interval $[0,1]$ is initially considered equally likely.

At each impression:

1. A user response is generated according to the true CTR.

The response is simulated as

$$
Y_k=
\begin{cases}
1, & U<\theta_{\text{true}},\\
0, & U\geq\theta_{\text{true}},
\end{cases}
$$

where

$$
U\sim\mathrm{Uniform}(0,1).
$$

2. The Beta distribution parameters are updated analytically:

$$
\alpha_k=\alpha_{k-1}+y_k,
$$

$$
\beta_k=\beta_{k-1}+(1-y_k).
$$

3. The two sequential estimators are calculated:

### Posterior Mean

$$
\hat{\theta}_{\mathrm{Bayes}}^{(k)}
=
\frac{\alpha_k}
{\alpha_k+\beta_k}.
$$


### MAP Estimate

$$
\hat{\theta}_{MAP}^{(k)}
=
\frac{\alpha_k-1}
{\alpha_k+\beta_k-2}.
$$

The evolution of these estimators is plotted from step $0$ to step $100$ together with the true CTR reference value.

In [4]:
import numpy as np
import plotly.graph_objects as go

# ----------------------------------------------------
# Bayesian CTR Tracking using Beta-Binomial Updates
# ----------------------------------------------------

# Reproducibility
np.random.seed(42)

# True hidden CTR
theta_true = 0.35

# Number of impressions
n_impressions = 100

# Initial Beta prior parameters
alpha = 1
beta = 1

# Storage arrays
bayes_estimates = [alpha / (alpha + beta)]
map_estimates = [0.0]

steps = list(range(n_impressions + 1))


# ----------------------------------------------------
# Sequential Updating
# ----------------------------------------------------

for k in range(n_impressions):

    # Generate user response
    random_value = np.random.uniform(0,1)

    if random_value < theta_true:
        y_k = 1
    else:
        y_k = 0


    # Analytical Beta-Binomial update
    alpha = alpha + y_k
    beta = beta + (1-y_k)


    # Posterior Mean
    theta_bayes = alpha / (alpha + beta)


    # MAP estimate
    if alpha > 1 and beta > 1:
        theta_map = (alpha - 1) / (alpha + beta - 2)

    elif alpha <= 1 and beta > 1:
        theta_map = 0

    elif beta <= 1 and alpha > 1:
        theta_map = 1

    else:
        theta_map = 0.5


    # Store estimates
    bayes_estimates.append(theta_bayes)
    map_estimates.append(theta_map)



# ----------------------------------------------------
# Plot Estimator Convergence
# ----------------------------------------------------

fig = go.Figure()


# True CTR reference line
fig.add_hline(
    y=theta_true,
    line_dash="dash",
    line_color="red",
    annotation_text="True CTR = 0.35",
    annotation_position="bottom right"
)


# Posterior Mean
fig.add_trace(
    go.Scatter(
        x=steps,
        y=bayes_estimates,
        mode="lines+markers",
        name="Posterior Mean (Bayes Estimate)"
    )
)


# MAP Estimate
fig.add_trace(
    go.Scatter(
        x=steps,
        y=map_estimates,
        mode="lines+markers",
        name="MAP Estimate"
    )
)


fig.update_layout(
    title="Sequential Bayesian Tracking of Advertisement CTR",
    xaxis_title="Number of Impressions (k)",
    yaxis_title="Estimated CTR (θ)",
    yaxis=dict(range=[0,1]),
    template="plotly_white"
)


fig.show()

# Analysis of Convergence

At the beginning of the simulation, the platform has no prior knowledge about the advertisement performance. Since the initial prior is

$$
\mathrm{Beta}(1,1),
$$

the initial posterior mean is

$$
E[\Theta]=
\frac{1}{1+1}
=
0.5.
$$

This value represents complete uncertainty rather than a measured estimate of the true CTR.

As impressions accumulate, each user interaction provides additional evidence. Clicks increase the value of $\alpha$, while non-clicks increase the value of $\beta$.

The posterior mean becomes

$$
\hat{\theta}_{\mathrm{Bayes}}
=
\frac{\alpha_0+C_k}
{\alpha_0+\beta_0+k},
$$

where $C_k$ represents the number of clicks observed after $k$ impressions.

As

$$
k\rightarrow\infty,
$$

the effect of the initial prior parameters becomes negligible:

$$
\frac{\alpha_0+C_k}
{\alpha_0+\beta_0+k}
\rightarrow
\frac{C_k}{k}.
$$

Therefore, the Bayesian estimate approaches the observed sample click-through rate.

The MAP estimate follows a similar behaviour. Initially, it may fluctuate more because the posterior distribution is strongly influenced by the prior and only a small number of observations are available. However, with increasing sample size, both the Posterior Mean and MAP estimates converge toward the true CTR,

$$
\theta_{\text{true}}=0.35.
$$

This demonstrates the key advantage of sequential Bayesian updating:

- early estimates incorporate prior knowledge to avoid extreme conclusions,
- additional user interactions gradually dominate the estimate,
- uncertainty decreases as more evidence becomes available.

Hence, increasing the number of impressions improves the accuracy and confidence of the platform's CTR estimation.